In [16]:
# load modules and define path
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import shap

PATH = '../df_for_training.csv'

## Data preparation


In [17]:
# load and view csv as dataframe
df = pd.read_csv(PATH)
df.head()

,id,age,sincere,intelligence,funny,ambition,decision_o,pref_o_sincere,pref_o_intelligence,pref_o_funny,...,both_like_movies,both_dislike_movies,both_like_concerts,both_dislike_concerts,both_like_music,both_dislike_music,both_like_shopping,both_dislike_shopping,both_like_yoga,both_dislike_yoga
0,0,28.0,8.0,9.0,8.0,9.0,0,22,22,12,...,0,1,0,1,0,1,0,1,0,1
1,1,23.0,8.0,10.0,8.0,9.0,0,22,22,12,...,0,1,0,1,1,1,0,1,0,1
2,2,26.0,7.0,8.0,8.0,7.0,0,22,22,12,...,0,1,0,1,0,1,0,1,0,1
3,3,21.0,7.0,8.0,9.0,8.0,1,22,22,12,...,0,1,0,1,1,1,0,1,0,1
4,4,24.0,9.0,10.0,8.0,8.0,0,22,22,12,...,0,1,0,1,1,1,0,1,0,1


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 48 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       30 non-null     int64  
 1   age                      30 non-null     float64
 2   sincere                  30 non-null     float64
 3   intelligence             30 non-null     float64
 4   funny                    30 non-null     float64
 5   ambition                 30 non-null     float64
 6   decision_o               30 non-null     int64  
 7   pref_o_sincere           30 non-null     int64  
 8   pref_o_intelligence      30 non-null     int64  
 9   pref_o_funny             30 non-null     int64  
 10  pref_o_ambitious         30 non-null     int64  
 11  pref_o_shared_interests  30 non-null     int64  
 12  d_age                    30 non-null     int64  
 13  interest_correlate       0 non-null      float64
 14  both_like_sports         30 non-null   

In [19]:
for col in df.columns:
    if df[col].dtype != "float64" :

        encode = OrdinalEncoder()
        encode.fit(df[[col]])

        df[col] = encode.fit_transform(df[[col]])
df.dropna(inplace = True)

In [20]:
# # drop 'match' and 'decision_o' column
# col_to_drop = ['match', 'decision_o', 'has_null', 'wave', 'expected_num_matches', 'd_expected_happy_with_sd_people', 'd_expected_num_matches', 'd_expected_num_interested_in_me', 'd_expected_num_matches', 'like', 'guess_prob_liked', 'd_like', 'd_guess_prob_liked', 'met']
# df.drop(col_to_drop, axis=1, inplace=True)

In [ ]:
interest_cols = ['d_sports', 'd_tvsports', 'd_exercise', 'd_dining', 'd_museums', 'd_art', 'd_hiking', 'd_gaming',
    'd_clubbing', 'd_reading', 'd_tv', 'd_theater', 'd_movies', 'd_concerts', 'd_music', 'd_shopping',
    'd_yoga']
col_to_use = [
    'age', 'd_age',
    'pref_o_sincere', 'pref_o_intelligence', 'pref_o_funny', 'pref_o_ambitious', 'pref_o_shared_interests',
    'sincere', 'intelligence', 'funny', 'ambition',
    'interests_correlate','decision_o',
] + interest_cols

df = df[col_to_use]
print(len(df))
df.head()

KeyError: "['gender', 'interests_correlate', 'd_sports', 'd_tvsports', 'd_exercise', 'd_dining', 'd_museums', 'd_art', 'd_hiking', 'd_gaming', 'd_clubbing', 'd_reading', 'd_tv', 'd_theater', 'd_movies', 'd_concerts', 'd_music', 'd_shopping', 'd_yoga'] not in index"

In [ ]:
# one hot encode 2x top both_like... and both_dislike... shared interestes
for col in interest_cols:
    like_name = col.replace('d_', 'both_like_')
    dislike_name = col.replace('d_', 'both_dislike_')
    df[like_name] = 0
    df[dislike_name] = 0

for row in df.itertuples():
    # make list of all values in interest_cols for this row
    interests = [getattr(row, col) for col in interest_cols]
    # get index of top 2 and bottom 2 values
    top_2_idx = sorted(range(len(interests)), key=lambda i: interests[i], reverse=True)[:2]
    bottom_2_idx = sorted(range(len(interests)), key=lambda i: interests[i])
    # get names of top_2_idx and bottom_2_idx
    top_2_cols = [interest_cols[i] for i in top_2_idx]
    bottom_2_cols = [interest_cols[i] for i in bottom_2_idx]
    # set like_name to 1 for top_2_cols and dislike_name to 1 for bottom_2_cols
    for col in top_2_cols:
        like_name = col.replace('d_', 'both_like_')
        df.at[row.Index, like_name] = 1
    for col in bottom_2_cols:
        dislike_name = col.replace('d_', 'both_dislike_')
        df.at[row.Index, dislike_name] = 1

# remove original interest columns
df.drop(interest_cols, axis=1, inplace=True)

KeyError: "['d_sports', 'd_tvsports', 'd_exercise', 'd_dining', 'd_museums', 'd_art', 'd_hiking', 'd_gaming', 'd_clubbing', 'd_reading', 'd_tv', 'd_theater', 'd_movies', 'd_concerts', 'd_music', 'd_shopping', 'd_yoga'] not found in axis"

In [ ]:
df.columns

Index(['id', 'age', 'sincere', 'intelligence', 'funny', 'ambition',
       'decision_o', 'pref_o_sincere', 'pref_o_intelligence', 'pref_o_funny',
       'pref_o_ambitious', 'pref_o_shared_interests', 'd_age',
       'interest_correlate', 'both_like_sports', 'both_dislike_sports',
       'both_like_tvsports', 'both_dislike_tvsports', 'both_like_exercise',
       'both_dislike_exercise', 'both_like_dining', 'both_dislike_dining',
       'both_like_museums', 'both_dislike_museums', 'both_like_art',
       'both_dislike_art', 'both_like_hiking', 'both_dislike_hiking',
       'both_like_gaming', 'both_dislike_gaming', 'both_like_clubbing',
       'both_dislike_clubbing', 'both_like_reading', 'both_dislike_reading',
       'both_like_tv', 'both_dislike_tv', 'both_like_theater',
       'both_dislike_theater', 'both_like_movies', 'both_dislike_movies',
       'both_like_concerts', 'both_dislike_concerts', 'both_like_music',
       'both_dislike_music', 'both_like_shopping', 'both_dislike_shopp

In [ ]:
[print(col) for col in df.columns];

id
age
sincere
intelligence
funny
ambition
decision_o
pref_o_sincere
pref_o_intelligence
pref_o_funny
pref_o_ambitious
pref_o_shared_interests
d_age
interest_correlate
both_like_sports
both_dislike_sports
both_like_tvsports
both_dislike_tvsports
both_like_exercise
both_dislike_exercise
both_like_dining
both_dislike_dining
both_like_museums
both_dislike_museums
both_like_art
both_dislike_art
both_like_hiking
both_dislike_hiking
both_like_gaming
both_dislike_gaming
both_like_clubbing
both_dislike_clubbing
both_like_reading
both_dislike_reading
both_like_tv
both_dislike_tv
both_like_theater
both_dislike_theater
both_like_movies
both_dislike_movies
both_like_concerts
both_dislike_concerts
both_like_music
both_dislike_music
both_like_shopping
both_dislike_shopping
both_like_yoga
both_dislike_yoga


In [ ]:
df.describe()

,id,age,sincere,intelligence,funny,ambition,decision_o,pref_o_sincere,pref_o_intelligence,pref_o_funny,...,both_like_movies,both_dislike_movies,both_like_concerts,both_dislike_concerts,both_like_music,both_dislike_music,both_like_shopping,both_dislike_shopping,both_like_yoga,both_dislike_yoga
count,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
X = df.drop('decision_o', axis=1)
y = df['decision_o']


KeyError: 'gender'

## Training

- I don't expect shap explanation to be very insightful, because the data is from different participants (as there is no way to distinguish them). What we might see is that common dating preferences appear in the explanations.
- TODO: the number of features still need to be decreased, by manual selection.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=500)
model.fit(X, y)
accuracy = model.score(X, y)
print(f'The accuracy is: {accuracy}')


The accuracy is: 0.5794392523364486


In [57]:
from sklearn.inspection import PartialDependenceDisplay
from sklearn.preprocessing import MinMaxScaler
import numpy as np

features = ['sincere',
            'intelligence',
            'funny',
            'ambition',
            'interests_correlate'      # tells us level of shared interest
            ]
features_idx = [X_w_train.columns.get_loc(feature) for feature in features]   
coeficients = []

for idx in features_idx:
    coeficients.append(float(model.coef_[0][idx]))
    print(f'Coefficient for feature {X_w_train.columns[idx]}: {model.coef_[0][idx]:.2f}')

# normalize coeficients between 0 and 1
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_coeficients = scaler.fit_transform(np.array(coeficients).reshape(-1, 1)).flatten()
normalized_coeficients = []
for idx, feature in enumerate(features):
    normalized_coeficient = scaled_coeficients[idx] / scaled_coeficients.sum() * 100
    normalized_coeficients.append(float(normalized_coeficient))
print(f'Normalized coefficients: {normalized_coeficients}')

Coefficient for feature sincere: -0.77
Coefficient for feature intelligence: -0.15
Coefficient for feature funny: 0.17
Coefficient for feature ambition: -0.65
Coefficient for feature interests_correlate: -0.18
Normalized coefficients: [0.0, 27.35038140135827, 41.33937473837728, 5.2677953740592, 26.042448486205245]


In [54]:
# not used anymore

PartialDependenceDisplay.from_estimator(model, X_test_sel, features_idx, kind='average')

ImportError: PartialDependenceDisplay.from_estimator requires matplotlib. You can install matplotlib with `pip install matplotlib`

In [ ]:
# not used anymore

explainer = shap.Explainer(model, X_train_sel)
shap_values = explainer(X_test_sel)
# shap.waterfall_plot(shap_values[0])
shap.summary_plot(shap_values, X_test_sel)
print(f'The accuracy of the explainer model is: {accuracy}')

## Other models (not used anymore)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_dt = DecisionTreeClassifier(random_state=42)
model_dt.fit(X_train_sel, y_train_sel)
accuracy_dt = model_dt.score(X_test_sel, y_test_sel)
print(f'The accuracy of decision tree is: {accuracy_dt:.3f}')

# print the feature importance of the decision tree model
feature_importance = model_dt.feature_importances_
if hasattr(X_train_sel, "columns"):
    feature_names = X_train_sel.columns
else:
    feature_names = X_w_train.columns  # feature order matches the NumPy array

for col, importance in zip(feature_names, feature_importance):
    if importance > 0:
        print(f"{col}: {importance:.2f}")

explainer = shap.Explainer(model_dt, X_train_sel)
shap_values = explainer(X_test_sel)
shap.summary_plot(shap_values, X_test_sel)
print(f'The accuracy of the explainer model is: {accuracy_dt:.3f}')

In [ ]:
# random forest
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train_sel, y_train_sel)
accuracy_rf = model_rf.score(X_test_sel, y_test_sel)
print(f'The accuracy of random forest is: {accuracy_rf:.3f}')

explainer = shap.Explainer(model_rf, X_train_sel,)
shap_values = explainer(X_test_sel, check_additivity=False)
shap.summary_plot(shap_values, X_test_sel)
print(f'The accuracy of the explainer model is: {accuracy_rf:.3f}')

In [ ]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(random_state=42)
model_xgb.fit(X_train_sel, y_train_sel)
accuracy_xgb = model_xgb.score(X_test_sel, y_test_sel)
print(f'The accuracy of xgboost is: {accuracy_xgb:.3f}')

explainer = shap.Explainer(model_xgb, X_train_sel)
shap_values = explainer(X_test_sel)
shap.summary_plot(shap_values, X_test_sel)
print(f'The accuracy of the explainer model is: {accuracy_xgb:.3f}')

### Notes
*using `X_w_train_mini` / `X_w_train_scaled_mini` and `y_w_train_mini` of size 60* using standard scaling

**Logistic model**
- The SHAP explainer model and the logistic model have the exact same accuracy, which means that the SHAP model probably works the same as the logistic model, so the explanations must be accurate.
- This model performed the best out of all, given that we use `X_w_train_scaled_mini`

**Decision tree /  Random Forest**
- The decision tree / Random Forest model is too simple for the problem at hand. It does not allow for enough complexity to get a decent accuracy, given a small dataset. We should not use this model.
- The SHAP explanation is terrible because the model is too simple.

**XGBoost**
- Better than logistic model

**Other**
- For non-black-box models, model native explanation might be better. Like regularized coefficients from the logistic model.
- All models achieve higher accuracy given more data, so this might be an issue in our project. However, is all data is from the same person (with the same preferences) than we probably will get a higher accuracy with less data.
- Use waterfall (instead of summary) plots to visualize the SHAP explanation for a single prediction.
